# Monitor Azure AI Foundry — Log Analytics Queries

Query a Log Analytics workspace to retrieve **AzureMetrics** data for Azure AI Foundry (Cognitive Services) resources.

Requirements:
- `azure-identity` and `azure-monitor-query` Python packages
- A Log Analytics workspace with diagnostic settings streaming **AllMetrics** from your Foundry instances
- Azure CLI login (`az login`) or another credential available to `DefaultAzureCredential`

Reference: [Monitor Azure AI Foundry models](https://learn.microsoft.com/en-us/azure/ai-foundry/foundry-models/how-to/monitor-models?view=foundry-classic)

In [ ]:
%pip install azure-identity azure-monitor-query pandas python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# id of the log analytics workspace where your foundry instance is sending logs to
# Find it in the Azure Portal → Log Analytics workspace → Properties → Workspace ID
WORKSPACE_ID = os.environ["WORKSPACE_ID"]
# location of your Cognitive Services account, e.g. "westus2", "westeurope", "swedencentral"
LOCATION = os.environ["LOCATION"]
# ── Set the full resource ID of your foundry instance ─────
# Format: /subscriptions/{sub}/resourceGroups/{rg}/providers/Microsoft.CognitiveServices/accounts/{name}
RESOURCE_ID = os.environ["RESOURCE_ID"]
# Azure Resource Manager API version
API_VERSION = os.environ["API_VERSION"]

In [ ]:
from datetime import timedelta

from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus
import pandas as pd

credential = DefaultAzureCredential()
client = LogsQueryClient(credential)

# ── Set your Log Analytics workspace ID here ────────────────────────

query = """
AzureMetrics
| union AzureDiagnostics
| take 2
"""

response = client.query_workspace(WORKSPACE_ID, query, timespan=timedelta(days=1))

if response.status == LogsQueryStatus.SUCCESS:
    table = response.tables[0]
    df = pd.DataFrame(data=table.rows, columns=table.columns)
    print("Query succeeded, here are the results:")
    display(df)
else:
    print("Query failed:", response.partial_error)

Query succeeded, here are the results:


,TenantId,SourceSystem,TimeGenerated,ResourceId,OperationName,OperationVersion,Category,ResultType,ResultSignature,ResultDescription,...,ManagementGroupName,Computer,RawData,AssetIdentity_g,EnqueueTime_t,event_s,location_s,properties_s,Tenant_s,AdditionalFields
0,4db52946-dd38-473b-bce5-abece098591d,Azure,2026-03-03 17:25:10.360000+00:00,/SUBSCRIPTIONS/0C15F8D8-6A36-40E4-8455-09841FE...,ChatCompletions_Create,,RequestResponse,,200,,...,,,,,2026-03-03 17:25:10.340000+00:00,ShoeboxCallResult,swedencentral,"{""apiName"":""OpenAI Language Model Instance API...",swedencentral,None
1,4db52946-dd38-473b-bce5-abece098591d,Azure,2026-03-03 20:16:59.201000+00:00,/SUBSCRIPTIONS/0C15F8D8-6A36-40E4-8455-09841FE...,Gets a list of assistants that were previously...,,RequestResponse,,200,,...,,,,,2026-03-03 20:16:59.135000+00:00,ShoeboxCallResult,swedencentral,"{""apiName"":""Azure AI Projects API"",""requestTim...",swedencentral,None


## Query Azure Monitor Metrics directly

Use `MetricsQueryClient` to query metrics straight from a Cognitive Services resource — no Log Analytics workspace required.

This queries the Azure Monitor Metrics API directly, which gives you near-real-time data with up to 1-minute granularity.

Reference: [azure-monitor-query MetricsQueryClient](https://learn.microsoft.com/en-us/python/api/azure-monitor-query/azure.monitor.query.metricsqueryclient)

In [12]:
%pip install azure-monitor-querymetrics --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from azure.monitor.querymetrics import MetricsClient, MetricAggregationType

# The MetricsClient needs the regional Azure Monitor endpoint for your resource's region.
# Format: https://<region>.metrics.monitor.azure.com
METRICS_ENDPOINT = f"https://{LOCATION}.metrics.monitor.azure.com"

metrics_client = MetricsClient(endpoint=METRICS_ENDPOINT, credential=credential)


### Total tokens (direct metrics, last 24h)

In [43]:
results = metrics_client.query_resources(
    resource_ids=[RESOURCE_ID],
    metric_namespace="Microsoft.CognitiveServices/accounts",
    metric_names=["TotalTokens", "ProcessedPromptTokens", "GeneratedTokens"],
    timespan=timedelta(hours=1),
    granularity=timedelta(hours=1),
    aggregations=[MetricAggregationType.TOTAL],
    filter="ModelDeploymentName eq '*'",  # Split by model deployment (like "Apply splitting" in the Portal)
)

rows = []
for result in results:
    for metric in result.metrics:
        for ts_element in metric.timeseries:
            deployment = ts_element.metadata_values.get("modeldeploymentname", "")
            for metric_value in ts_element.data:
                if metric_value.total is not None:
                    rows.append({
                        "Timestamp": metric_value.timestamp,
                        "Metric": metric.name,
                        "ModelDeploymentName": deployment,
                        "Total": metric_value.total,
                    })

df_tokens = pd.DataFrame(rows)

print("Metrics query succeeded, here are the results:")
print("Metrics for resource:", RESOURCE_ID)
display(df_tokens)

Metrics query succeeded, here are the results:
Metrics for resource: /subscriptions/00000000-0000-0000-0000-000000000000/resourceGroups/ai-hub-projects/providers/Microsoft.CognitiveServices/accounts/olid-foundry-swc


,Timestamp,Metric,ModelDeploymentName,Total
0,2026-03-04 13:04:00+00:00,TotalTokens,gpt-4.1-mini,27
1,2026-03-04 13:04:00+00:00,TotalTokens,phi-4,0
2,2026-03-04 13:04:00+00:00,TotalTokens,gpt-4o,0
3,2026-03-04 13:04:00+00:00,TotalTokens,text-embedding-3-small,0
4,2026-03-04 13:04:00+00:00,TotalTokens,mai-ds-r1,0
5,2026-03-04 13:04:00+00:00,TotalTokens,text-embedding-3-large,0
6,2026-03-04 13:04:00+00:00,TotalTokens,grok-3,0
7,2026-03-04 13:04:00+00:00,TotalTokens,gpt-5,0
8,2026-03-04 13:04:00+00:00,TotalTokens,gpt-4o-mini,0
9,2026-03-04 13:04:00+00:00,TotalTokens,o3-mini,0


### Same data from AzureMetrics in Log Analytics

Check whether the same per-deployment token breakdown is available via the `AzureMetrics` table in the Log Analytics workspace.

In [49]:
# Compare: AzureMetrics in Log Analytics vs direct Metrics API
# First, see which token-related metrics are available
query = """
AzureMetrics
| where ResourceProvider == "MICROSOFT.COGNITIVESERVICES"
| where MetricName has "Token" or MetricName in ("ProcessedPromptTokens", "GeneratedTokens")
| summarize
    Total = sum(Total),
    Rows = count(),
    MinTime = min(TimeGenerated),
    MaxTime = max(TimeGenerated)
    by Resource, MetricName
| order by Resource asc, MetricName asc
"""

response = client.query_workspace(WORKSPACE_ID, query, timespan=timedelta(days=7))

if response.status == LogsQueryStatus.SUCCESS:
    table = response.tables[0]
    df_la_tokens = pd.DataFrame(data=table.rows, columns=table.columns)
    print(f"Token metrics in AzureMetrics (Log Analytics) — {len(df_la_tokens)} rows:")
    display(df_la_tokens)
    
    if len(df_la_tokens) == 0:
        print("\n⚠ No token metrics found in AzureMetrics.")
        print("This means the diagnostic settings are NOT streaming token metrics to Log Analytics.")
        print("Only the direct Azure Monitor Metrics API has this data.")
else:
    print("Query failed:", response.partial_error)

Token metrics in AzureMetrics (Log Analytics) — 6 rows:


,Resource,MetricName,Total,Rows,MinTime,MaxTime
0,OLID-AI-GW-FOUNDRY-SWC-MODELS,GeneratedTokens,35794,8,2026-03-03 16:39:00+00:00,2026-03-04 08:04:00+00:00
1,OLID-AI-GW-FOUNDRY-SWC-MODELS,ProcessedPromptTokens,564,8,2026-03-03 16:39:00+00:00,2026-03-04 08:04:00+00:00
2,OLID-FOUNDRY-EASTUS2,GeneratedTokens,2394,1,2026-03-04 08:03:00+00:00,2026-03-04 08:03:00+00:00
3,OLID-FOUNDRY-EASTUS2,ProcessedPromptTokens,100,1,2026-03-04 08:03:00+00:00,2026-03-04 08:03:00+00:00
4,OLID-FOUNDRY-SWC,GeneratedTokens,18516,5,2026-03-03 17:21:00+00:00,2026-03-04 12:33:00+00:00
5,OLID-FOUNDRY-SWC,ProcessedPromptTokens,2253,5,2026-03-03 17:21:00+00:00,2026-03-04 12:33:00+00:00


### AzureDiagnostics — per-request detail

`AzureDiagnostics` contains one row per API call, with deployment name, model name, and request/response sizes.

In [51]:
# AzureDiagnostics — per-request fields for Cognitive Services
query = """
AzureDiagnostics
| where ResourceProvider == "MICROSOFT.COGNITIVESERVICES"
| where TimeGenerated > ago(1d)
| extend props = parse_json(properties_s)
| extend
    deployment = tostring(props.modelDeploymentName),
    model = tostring(props.modelName),
    operation = OperationName,
    reqBytes = tolong(props.requestLength),
    respBytes = tolong(props.responseLength),
    statusCode = tostring(props.statusCode),
    apiName = tostring(props.apiName),
    region = tostring(props.modelRegion)
| project TimeGenerated, Resource, operation, deployment, model, region, statusCode, reqBytes, respBytes, apiName, properties_s
| take 10
"""

response = client.query_workspace(WORKSPACE_ID, query, timespan=timedelta(days=1))

if response.status == LogsQueryStatus.SUCCESS:
    table = response.tables[0]
    df_diag = pd.DataFrame(data=table.rows, columns=table.columns)
    print(f"AzureDiagnostics — {len(df_diag)} sample rows")
    display(df_diag.drop(columns=["properties_s"]))
    
    if len(df_diag) > 0:
        print("\n── Raw properties_s for first row ──")
        import json
        print(json.dumps(json.loads(df_diag.iloc[0]["properties_s"]), indent=2))
    else:
        print("\n⚠ No AzureDiagnostics rows found. RequestResponse logs may not be enabled in diagnostic settings.")
else:
    print("Query failed:", response.partial_error)

AzureDiagnostics — 10 sample rows


,TimeGenerated,Resource,operation,deployment,model,region,statusCode,reqBytes,respBytes,apiName
0,2026-03-03 17:22:26.516000+00:00,OLID-FOUNDRY-EASTUS2,ChatCompletions_Create,gpt-5,gpt-5,,,119,1274,OpenAI Language Model Instance API
1,2026-03-03 17:23:27.754000+00:00,OLID-FOUNDRY-SWC,Creates an embedding vector representing the i...,text-embedding-3-small,text-embedding-3-small,,,83,33273,OpenAI Language Model Instance API
2,2026-03-03 17:24:12.594000+00:00,OLID-FOUNDRY-SWC,ChatCompletions_Create,MAI-DS-R1,MAI-DS-R1,,,77,0,OpenAI Language Model Instance API
3,2026-03-03 17:24:12.597000+00:00,OLID-FOUNDRY-SWC,ChatCompletions_Create,Phi-4,Phi-4,,,73,0,OpenAI Language Model Instance API
4,2026-03-03 17:24:34.978000+00:00,OLID-FOUNDRY-SWC,ChatCompletions_Create,Phi-4,Phi-4,,,73,0,OpenAI Language Model Instance API
5,2026-03-03 20:17:24.077000+00:00,OLID-AI-GW-FOUNDRY-SWC-MODELS,Projects_Wildcard_Get,,,,,0,17,Azure AI Projects API
6,2026-03-03 21:08:02.733000+00:00,OLID-FOUNDRY-SWC,List_Vector_Stores,,,,,0,96,Azure OpenAI API version 2025-03-01-preview
7,2026-03-03 21:08:25.665000+00:00,OLID-FOUNDRY-EASTUS2,Projects_Wildcard_Get,,,,,0,0,Azure AI Projects API
8,2026-03-04 02:16:24.749000+00:00,OLID-FOUNDRY-NCUS,Projects_Wildcard_Get,,,,,0,17,Azure AI Projects API
9,2026-03-04 02:16:36.839000+00:00,OLID-AI-GW-FOUNDRY-NO-MODELS,List_Assistants,,,,,0,96,Azure OpenAI API version 2025-03-01-preview



── Raw properties_s for first row ──
{
  "apiName": "OpenAI Language Model Instance API",
  "requestTime": 639081552357213903,
  "requestLength": 119,
  "responseTime": 639081552409142424,
  "responseLength": 1274,
  "objectId": "9f872b16-52de-493b-8353-9eae425b08c8",
  "streamType": "Non-Streaming",
  "modelDeploymentName": "gpt-5",
  "modelName": "gpt-5",
  "modelVersion": "2025-08-07"
}


### Quota & deployment allocation

Query the subscription-level quota limits per model/region, and the TPM allocation per deployment across all Cognitive Services accounts.

In [ ]:
import requests

SUBSCRIPTION_ID = RESOURCE_ID.split("/")[2]

token = credential.get_token("https://management.azure.com/.default").token
headers = {"Authorization": f"Bearer {token}"}

# ── 1. Subscription-level quota per model in this region ────────────
usages_url = (
    f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}"
    f"/providers/Microsoft.CognitiveServices/locations/{LOCATION}"
    f"/usages?api-version={API_VERSION}"
)
usages_resp = requests.get(usages_url, headers=headers)
usages_resp.raise_for_status()

usages = usages_resp.json().get("value", [])
quota_rows = []
for u in usages:
    if u.get("currentValue", 0) > 0 or "OpenAI" in u.get("name", {}).get("value", ""):
        quota_rows.append({
            "Model": u["name"]["value"],
            "Allocated (current)": u["currentValue"],
            "Limit": u["limit"],
            "Unit": u.get("unit", ""),
        })

df_quota = pd.DataFrame(quota_rows)
print(f"Subscription quota in {LOCATION} — {len(df_quota)} model SKUs with allocation:")
display(df_quota)

# ── 2. Per-deployment allocation across all accounts ────────────────
# List all Cognitive Services accounts in the subscription
accounts_url = (
    f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}"
    f"/providers/Microsoft.CognitiveServices/accounts"
    f"?api-version={API_VERSION}"
)
accounts_resp = requests.get(accounts_url, headers=headers)
accounts_resp.raise_for_status()

deploy_rows = []
for acct in accounts_resp.json().get("value", []):
    acct_id = acct["id"]
    acct_name = acct["name"]
    acct_location = acct["location"]
    # List deployments for this account
    deploys_url = (
        f"https://management.azure.com{acct_id}"
        f"/deployments?api-version={API_VERSION}"
    )
    deploys_resp = requests.get(deploys_url, headers=headers)
    if deploys_resp.status_code != 200:
        continue
    for d in deploys_resp.json().get("value", []):
        props = d.get("properties", {})
        model = props.get("model", {})
        sku = d.get("sku", {})
        rate_limits = props.get("rateLimits", [])
        tpm = next((r["count"] for r in rate_limits if r.get("key") == "token"), sku.get("capacity"))
        rpm = next((r["count"] for r in rate_limits if r.get("key") == "request"), None)
        deploy_rows.append({
            "Account": acct_name,
            "Location": acct_location,
            "Deployment": d["name"],
            "Model": model.get("name", ""),
            "Version": model.get("version", ""),
            "SKU": sku.get("name", ""),
            "Capacity (TPM K)": sku.get("capacity", ""),
            "TPM (rate limit)": tpm,
            "RPM (rate limit)": rpm,
        })

df_deployments = pd.DataFrame(deploy_rows)
print(f"\nDeployments across all accounts — {len(df_deployments)} total:")
display(df_deployments)

Subscription quota in swedencentral — 143 model SKUs with allocation:


,Model,Allocated (current),Limit,Unit
0,OpenAI.ProvisionedManaged,0,0,Count
1,OpenAI.GlobalProvisionedManaged,0,0,Count
2,OpenAI.Standard.Dalle,2,2,Count
3,OpenAI.FineTuned.Deployments,0,10,Count
4,OpenAI.S0.AccountCount,1,30,Count
...,...,...,...,...
138,AIServices.GlobalStandard.MAI-DS-R1,250,1000,Count
139,AIServices.GlobalStandard.mistral-document-ai-...,30,60,Count
140,AIServices.GlobalStandard.grok-3,150,1000,Count
141,AIServices.GlobalStandard.DeepSeek-V3.2,500,1000,Count



Deployments across all accounts — 48 total:


,Account,Location,Deployment,Model,Version,SKU,Capacity (TPM K),TPM (rate limit),RPM (rate limit)
0,oli-openai-eastus,eastus,gpt4o-mini,gpt-4o-mini,2024-07-18,Standard,100,100000,1000.0
1,oli-openai-eastus,eastus,gpt-4o,gpt-4o,2024-05-13,GlobalStandard,13,13000,13.0
2,oli-openai-eastus,eastus,text-embedding-3-small,text-embedding-3-small,1,Standard,120,120000,120.0
3,oli-openai-eastus,eastus,gpt-4o-mini-audio-preview,gpt-4o-mini-audio-preview,2024-12-17,GlobalStandard,250,250000,250.0
4,oli-openai-eastus,eastus,gpt-5-chn,gpt-5,2025-08-07,GlobalStandard,150,150000,1500.0
5,olid-foundry-eastus2,eastus2,gpt-5,gpt-5,2025-08-07,GlobalStandard,500,500000,5000.0
6,oli-openai-fra,francecentral,gpt4,gpt-4o,2024-11-20,Standard,10,10000,10.0
7,oli-openai-fra,francecentral,gpt4-32,gpt-4o,2024-11-20,Standard,30,30000,30.0
8,oli-openai-swe,swedencentral,gpt4o,gpt-4o,2024-05-13,Standard,100,100000,100.0
9,oli-openai-swe,swedencentral,gpt4-32,gpt-4o,2024-11-20,Standard,30,30000,30.0
